# CHORUS Model Specification

CHORUS is a **synthetic explanatory social simulation**. It represents how locally plausible decisions, uneven information, social relationships, platform conditions, and institutional constraints can combine into network-scale changes in interpretation and action.

This document makes the model legible as a research object. It does not convert the game into a behavioral forecast, diagnostic instrument, lie detector, trust score, or empirical estimate of any population.

In [1]:
from html import escape
from hashlib import sha256
import re

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

DIAGRAM_STYLE = r"""
.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}
"""

SVG_COLORS = {
    "canvas": "#06130c",
    "group_fill": "#0b2819",
    "group_stroke": "#799886",
    "person_fill": "#163e2a",
    "person_stroke": "#a7e0bd",
    "system_fill": "#103522",
    "system_stroke": "#e2c57f",
    "component_fill": "#0d281a",
    "component_stroke": "#9bc8aa",
    "data_fill": "#262b22",
    "data_stroke": "#d5dedc",
    "evidence_fill": "#282433",
    "evidence_stroke": "#c9b6db",
    "boundary_fill": "#2d271b",
    "boundary_stroke": "#e2c57f",
    "risk_fill": "#351f1f",
    "risk_stroke": "#e0a8a8",
    "decision_fill": "#352b18",
    "decision_stroke": "#e2c57f",
    "title": "#f2f1e8",
    "body": "#c7d1c9",
    "role": "#a7e0bd",
    "flow": "#e2c57f",
    "data": "#a7e0bd",
    "evidence": "#c9b6db",
    "boundary": "#d5dedc",
    "association": "#b8c0bc",
    "label_bg": "#07140d",
    "label_stroke": "#5a6c60",
}


def dnode(node_id, x, y, w, h, title, body="", kind="component", shape="rect", role=""):
    return {
        "id": node_id, "x": float(x), "y": float(y), "w": float(w), "h": float(h),
        "title": title, "body": body, "kind": kind, "shape": shape, "role": role,
    }


def dedge(source, target, points, kind="flow", label="", label_at=None, arrow=True):
    return {
        "source": source, "target": target,
        "points": tuple((float(x), float(y)) for x, y in points),
        "kind": kind, "label": label, "label_at": label_at, "arrow": arrow,
    }


def dgroup(group_id, x, y, w, h, label, kind="boundary"):
    return {
        "id": group_id, "x": float(x), "y": float(y), "w": float(w),
        "h": float(h), "label": label, "kind": kind,
    }


def _slug(value):
    cleaned = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return cleaned or "diagram"


def _lines(value, max_chars, max_lines=4):
    raw_lines = str(value).split("\n") if value else []
    lines = []
    for raw in raw_lines:
        words = raw.split()
        if not words:
            lines.append("")
            continue
        current = words[0]
        for word in words[1:]:
            candidate = current + " " + word
            if len(candidate) <= max_chars:
                current = candidate
            else:
                lines.append(current)
                current = word
        lines.append(current)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(" …") + "…"
    return lines


def _segments(points):
    return list(zip(points, points[1:]))


def _on_boundary(point, node, tolerance=0.01):
    x, y = point
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    on_vertical = (
        (abs(x - left) <= tolerance or abs(x - right) <= tolerance)
        and top - tolerance <= y <= bottom + tolerance
    )
    on_horizontal = (
        (abs(y - top) <= tolerance or abs(y - bottom) <= tolerance)
        and left - tolerance <= x <= right + tolerance
    )
    return on_vertical or on_horizontal


def _segment_axis(segment):
    (x1, y1), (x2, y2) = segment
    if x1 == x2 and y1 != y2:
        return "v"
    if y1 == y2 and x1 != x2:
        return "h"
    raise AssertionError(f"Diagram route segment must be orthogonal and nonzero: {segment}")


def _segment_crosses_rect(segment, node):
    (x1, y1), (x2, y2) = segment
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    axis = _segment_axis(segment)
    if axis == "h":
        if not top < y1 < bottom:
            return False
        return max(min(x1, x2), left) < min(max(x1, x2), right)
    if not left < x1 < right:
        return False
    return max(min(y1, y2), top) < min(max(y1, y2), bottom)


def _segment_intersection(first, second):
    a1, a2 = first
    b1, b2 = second
    axis_a, axis_b = _segment_axis(first), _segment_axis(second)
    if axis_a != axis_b:
        horizontal = first if axis_a == "h" else second
        vertical = second if axis_a == "h" else first
        (hx1, hy), (hx2, _) = horizontal
        (vx, vy1), (_, vy2) = vertical
        if min(hx1, hx2) <= vx <= max(hx1, hx2) and min(vy1, vy2) <= hy <= max(vy1, vy2):
            return (vx, hy)
        return None
    if axis_a == "h" and a1[1] == b1[1]:
        lo = max(min(a1[0], a2[0]), min(b1[0], b2[0]))
        hi = min(max(a1[0], a2[0]), max(b1[0], b2[0]))
        if lo < hi:
            return ("overlap", lo, hi, a1[1])
        if lo == hi:
            return (lo, a1[1])
    if axis_a == "v" and a1[0] == b1[0]:
        lo = max(min(a1[1], a2[1]), min(b1[1], b2[1]))
        hi = min(max(a1[1], a2[1]), max(b1[1], b2[1]))
        if lo < hi:
            return ("overlap", lo, hi, a1[0])
        if lo == hi:
            return (a1[0], lo)
    return None


def _rectangles_overlap(first, second):
    return (
        max(first["x"], second["x"]) < min(first["x"] + first["w"], second["x"] + second["w"])
        and max(first["y"], second["y"]) < min(first["y"] + first["h"], second["y"] + second["h"])
    )


def _validate_diagram(width, height, nodes, edges):
    assert width > 0 and height > 0
    node_map = {node["id"]: node for node in nodes}
    assert len(node_map) == len(nodes), "Diagram node IDs must be unique."
    for node in nodes:
        assert node["w"] > 0 and node["h"] > 0
        assert 0 <= node["x"] < width and 0 <= node["y"] < height
        assert node["x"] + node["w"] <= width and node["y"] + node["h"] <= height
    for index, first in enumerate(nodes):
        for second in nodes[index + 1:]:
            assert not _rectangles_overlap(first, second), (
                f"Diagram nodes overlap: {first['id']} and {second['id']}"
            )

    all_segments = []
    for edge_index, edge in enumerate(edges):
        assert edge["source"] in node_map and edge["target"] in node_map
        points = edge["points"]
        assert len(points) >= 2
        assert _on_boundary(points[0], node_map[edge["source"]]), (
            f"Route must start on source boundary: {edge}"
        )
        assert _on_boundary(points[-1], node_map[edge["target"]]), (
            f"Route must end on target boundary: {edge}"
        )
        for segment_index, segment in enumerate(_segments(points)):
            _segment_axis(segment)
            for node_id, node in node_map.items():
                if node_id in (edge["source"], edge["target"]):
                    continue
                assert not _segment_crosses_rect(segment, node), (
                    f"Route crosses node {node_id}: {edge}"
                )
            all_segments.append((edge_index, segment_index, edge, segment))

    for index, first in enumerate(all_segments):
        for second in all_segments[index + 1:]:
            edge_a, edge_b = first[2], second[2]
            if first[0] == second[0]:
                continue
            intersection = _segment_intersection(first[3], second[3])
            if intersection is None:
                continue
            shared_terminal_points = (
                set((edge_a["points"][0], edge_a["points"][-1]))
                & set((edge_b["points"][0], edge_b["points"][-1]))
            )
            if (
                isinstance(intersection, tuple)
                and intersection
                and intersection[0] != "overlap"
                and intersection in shared_terminal_points
            ):
                continue
            raise AssertionError(
                f"Diagram routes cross or overlap at {intersection}: {edge_a} / {edge_b}"
            )
    return {
        "nodes": len(nodes), "edges": len(edges), "segments": len(all_segments),
        "crossings": 0, "node_incursions": 0, "node_overlaps": 0,
    }


def _svg_text(x, y, lines, fill, size, weight=400, line_height=15, anchor="start", letter_spacing=0):
    if not lines:
        return ""
    spans = []
    for index, line in enumerate(lines):
        dy = 0 if index == 0 else line_height
        spans.append(f'<tspan x="{x:g}" dy="{dy:g}">{escape(line)}</tspan>')
    return (
        f'<text x="{x:g}" y="{y:g}" fill="{fill}" font-size="{size:g}" '
        f'font-weight="{weight}" text-anchor="{anchor}" letter-spacing="{letter_spacing:g}" '
        'font-family="Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">'
        f'{"".join(spans)}</text>'
    )


def _node_colors(kind):
    return (
        SVG_COLORS.get(f"{kind}_fill", SVG_COLORS["component_fill"]),
        SVG_COLORS.get(f"{kind}_stroke", SVG_COLORS["component_stroke"]),
    )


def _edge_dash(kind):
    if kind == "evidence":
        return ' stroke-dasharray="7 5"'
    if kind == "boundary":
        return ' stroke-dasharray="3 5"'
    return ""


def _figure_shell(diagram_type, title, description, svg, assurance, legend=(), notes=(), equivalent=""):
    legend_html = ""
    if legend:
        legend_items = ''.join(
            f'<li><i class="diagram-key diagram-key-{escape(kind)}" aria-hidden="true"></i>'
            f'<span>{escape(label)}</span></li>'
            for kind, label in legend
        )
        legend_html = f'<ul class="diagram-legend" aria-label="Diagram legend">{legend_items}</ul>'
    notes_html = ''.join(f'<li>{escape(str(note))}</li>' for note in notes)
    if notes_html:
        notes_html = f'<ul class="diagram-notes">{notes_html}</ul>'
    return HTMLResult(
        f'<figure class="diagram-figure" data-diagram-type="{escape(diagram_type)}" '
        'data-routing="orthogonal-crossing-free">'
        f'<figcaption><span>{escape(diagram_type)}</span><strong>{escape(title)}</strong>'
        f'<p>{escape(description)}</p></figcaption>'
        f'<div class="diagram-canvas" role="region" aria-label="{escape(title)} diagram" tabindex="0">{svg}</div>'
        f'{legend_html}{notes_html}<span class="diagram-assurance">{escape(assurance)}</span>{equivalent}'
        '</figure>'
    )


def diagram_html(diagram_type, title, description, width, height, nodes, edges, groups=(), legend=(), notes=()):
    nodes = tuple(nodes)
    edges = tuple(edges)
    groups = tuple(groups)
    assurance = _validate_diagram(width, height, nodes, edges)
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    node_map = {node["id"]: node for node in nodes}

    marker_kinds = sorted(set(edge.get("kind", "flow") for edge in edges if edge.get("arrow", True)))
    defs = []
    for kind in marker_kinds:
        marker = uid + "-arrow-" + _slug(kind)
        color = SVG_COLORS.get(kind, SVG_COLORS["flow"])
        defs.append(
            f'<marker id="{marker}" viewBox="0 0 10 10" refX="9" refY="5" '
            'markerWidth="7" markerHeight="7" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}"/></marker>'
        )

    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" '
        f'viewBox="0 0 {width:g} {height:g}" role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
        f'<defs>{"".join(defs)}</defs>',
    ]

    for group in groups:
        parts.append(
            f'<rect x="{group["x"]:g}" y="{group["y"]:g}" width="{group["w"]:g}" '
            f'height="{group["h"]:g}" rx="18" fill="{SVG_COLORS["group_fill"]}" fill-opacity=".28" '
            f'stroke="{SVG_COLORS["group_stroke"]}" stroke-width="1.2" stroke-dasharray="7 5"/>'
        )
        parts.append(
            _svg_text(group["x"] + 14, group["y"] + 21, [group["label"]], SVG_COLORS["title"], 12, 700, 14, "start", .7)
        )

    for edge in edges:
        points = " ".join(f'{x:g},{y:g}' for x, y in edge["points"])
        color = SVG_COLORS.get(edge["kind"], SVG_COLORS["flow"])
        marker_attr = ""
        if edge.get("arrow", True):
            marker_attr = f' marker-end="url(#{uid}-arrow-{_slug(edge["kind"])})"'
        parts.append(
            f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2" '
            f'stroke-linecap="round" stroke-linejoin="round"{_edge_dash(edge["kind"])}{marker_attr}/>'
        )
        if edge.get("label"):
            lx, ly = edge.get("label_at") or edge["points"][len(edge["points"]) // 2]
            label_width = max(72, min(150, len(edge["label"]) * 6.4 + 18))
            parts.append(
                f'<rect x="{lx - label_width / 2:g}" y="{ly - 12:g}" width="{label_width:g}" height="22" '
                f'rx="7" fill="{SVG_COLORS["label_bg"]}" stroke="{SVG_COLORS["label_stroke"]}" stroke-width=".8"/>'
            )
            parts.append(_svg_text(lx, ly + 3, [edge["label"]], SVG_COLORS["title"], 10, 700, 12, "middle"))

    for node in nodes:
        x, y, w, h = node["x"], node["y"], node["w"], node["h"]
        shape = node.get("shape", "rect")
        fill, stroke = _node_colors(node.get("kind", "component"))
        common = f'fill="{fill}" stroke="{stroke}" stroke-width="1.6"'
        if shape == "diamond":
            points = f'{x + w / 2:g},{y:g} {x + w:g},{y + h / 2:g} {x + w / 2:g},{y + h:g} {x:g},{y + h / 2:g}'
            parts.append(f'<polygon points="{points}" {common}/>' )
        elif shape == "pill":
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="{h / 2:g}" {common}/>' )
        elif shape == "document":
            fold = min(18, w * .12)
            d = (
                f'M {x:g} {y:g} H {x + w - fold:g} L {x + w:g} {y + fold:g} '
                f'V {y + h:g} H {x:g} Z M {x + w - fold:g} {y:g} V {y + fold:g} H {x + w:g}'
            )
            parts.append(f'<path d="{d}" {common} stroke-linejoin="round"/>' )
        else:
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="12" {common}/>' )

        char_width = max(13, int((w - 24) / 7.1))
        title_lines = _lines(node["title"], char_width, 2)
        body_lines = _lines(node.get("body", ""), char_width, 4)
        role = node.get("role", "")
        top = y + 20
        if role:
            parts.append(_svg_text(x + w / 2, top, [role.upper()], SVG_COLORS["role"], 10, 700, 12, "middle", .8))
            top += 18
        parts.append(_svg_text(x + w / 2, top, title_lines, SVG_COLORS["title"], 14, 700, 16, "middle"))
        body_y = top + 16 * len(title_lines) + 5
        parts.append(_svg_text(x + w / 2, body_y, body_lines, SVG_COLORS["body"], 11.5, 400, 14, "middle"))

    parts.append('</svg>')
    relation_items = []
    for edge in edges:
        relation = f'{node_map[edge["source"]]["title"]} → {node_map[edge["target"]]["title"]}'
        if edge.get("label"):
            relation += f' ({edge["label"]})'
        relation_items.append(f'<li>{escape(relation)}</li>')
    node_items = [
        f'<li><strong>{escape(node["title"])}</strong>'
        f'{": " + escape(node["body"]) if node.get("body") else ""}</li>'
        for node in nodes
    ]
    equivalent = (
        '<details class="diagram-equivalent"><summary>Text equivalent</summary>'
        f'<h4>Elements</h4><ul>{"".join(node_items)}</ul>'
        f'<h4>Relationships</h4><ul>{"".join(relation_items) if relation_items else "<li>No connector relationships; the diagram uses nested evidentiary zones.</li>"}</ul>'
        '</details>'
    )
    assurance_text = (
        f'Validated: {assurance["nodes"]} nodes · {assurance["edges"]} edges · '
        'orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps'
    )
    return _figure_shell(
        diagram_type, title, description, ''.join(parts), assurance_text,
        legend=legend, notes=notes, equivalent=equivalent,
    )


def matrix_diagram_html(diagram_type, title, description, rows, columns, coverage, notes=()):
    rows = tuple(rows)
    columns = tuple(columns)
    valid_marks = {"P", "S", ""}
    assert len(set(rows)) == len(rows) and len(set(columns)) == len(columns)
    for key, mark in coverage.items():
        assert key[0] in rows and key[1] in columns and mark in valid_marks

    left = 255
    top = 125
    cell_w = 125
    cell_h = 72
    right_pad = 25
    bottom_pad = 35
    width = left + cell_w * len(columns) + right_pad
    height = top + cell_h * len(rows) + bottom_pad
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" viewBox="0 0 {width:g} {height:g}" '
        f'role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
    ]
    for column_index, column in enumerate(columns):
        x = left + column_index * cell_w
        lines = _lines(column, 15, 3)
        parts.append(_svg_text(x + cell_w / 2, 38, lines, SVG_COLORS["title"], 11, 700, 14, "middle"))
    for row_index, row in enumerate(rows):
        y = top + row_index * cell_h
        parts.append(
            f'<rect x="8" y="{y:g}" width="{left - 16:g}" height="{cell_h:g}" rx="8" '
            f'fill="{SVG_COLORS["component_fill"]}" stroke="{SVG_COLORS["component_stroke"]}" stroke-width="1"/>'
        )
        parts.append(_svg_text(20, y + 28, _lines(row, 30, 2), SVG_COLORS["title"], 12, 700, 15, "start"))
        for column_index, column in enumerate(columns):
            x = left + column_index * cell_w
            mark = coverage.get((row, column), "")
            if mark == "P":
                fill, stroke, label = "#173d29", "#a7e0bd", "P"
            elif mark == "S":
                fill, stroke, label = "#2b2835", "#c9b6db", "S"
            else:
                fill, stroke, label = "#0a1a11", "#38483e", "—"
            parts.append(
                f'<rect x="{x:g}" y="{y:g}" width="{cell_w:g}" height="{cell_h:g}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1"/>'
            )
            parts.append(_svg_text(x + cell_w / 2, y + 42, [label], SVG_COLORS["title"] if mark else SVG_COLORS["body"], 17, 700, 18, "middle"))
    parts.append('</svg>')

    table_rows = []
    for row in rows:
        table_rows.append((row, *({"P": "Primary", "S": "Supporting", "": "Not claimed"}[coverage.get((row, column), "")] for column in columns)))
    equivalent = str(table_html(
        title + " text equivalent",
        ("Test family", *columns),
        table_rows,
        row_headers=True,
    ))
    equivalent = f'<details class="diagram-equivalent"><summary>Text equivalent</summary>{equivalent}</details>'
    return _figure_shell(
        diagram_type, title, description, ''.join(parts),
        f'Validated: {len(rows)} test families · {len(columns)} concern columns · matrix topology · 0 connector lines',
        legend=(("data", "P = primary coverage"), ("evidence", "S = supporting coverage")),
        notes=notes,
        equivalent=equivalent,
    )


_html = HTMLResult(f"<style>{DIAGRAM_STYLE}</style>")
print("Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.")
_html

Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.


.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}

## Claim boundary

The simulation can establish consequences **inside its authored rules**. It cannot establish the prevalence, probability, effect size, or causal structure of real-world disinformation behavior without external data, calibration, and validation.

In [2]:
claims = [
    ("System behavior", "Supported", "Given a seed, state, and accepted action, the reducer produces the documented deterministic receipts."),
    ("Mechanism illustration", "Supported with limits", "The model can show how a specified mechanism behaves under its own assumptions."),
    ("Comparative variant", "Supported within-model", "Matched variants can isolate the consequences of changing one authored condition."),
    ("Human diagnosis", "Not supported", "A fictional interior state cannot be used to infer the motive, pathology, or honesty of a real person."),
    ("Population estimate", "Not supported", "Synthetic frequencies are generated by grammar and policy, not sampled from a population."),
    ("Real-world forecast", "Not supported", "Reach, trust, fatigue, and belief outputs are not calibrated predictive probabilities."),
    ("Intervention efficacy", "Not supported externally", "An intervention that works inside CHORUS requires empirical evaluation before any real-world claim."),
]
_html = table_html("CHORUS claim ladder", ("Claim class", "Status", "Meaning"), claims, row_headers=True)
assert sum(status.startswith("Not supported") for _, status, _ in claims) == 4
print("PASS: claim ladder separates internal model evidence from external empirical claims.")
_html

PASS: claim ladder separates internal model evidence from external empirical claims.


Claim class,Status,Meaning
System behavior,Supported,"Given a seed, state, and accepted action, the reducer produces the documented deterministic receipts."
Mechanism illustration,Supported with limits,The model can show how a specified mechanism behaves under its own assumptions.
Comparative variant,Supported within-model,Matched variants can isolate the consequences of changing one authored condition.
Human diagnosis,Not supported,"A fictional interior state cannot be used to infer the motive, pathology, or honesty of a real person."
Population estimate,Not supported,"Synthetic frequencies are generated by grammar and policy, not sampled from a population."
Real-world forecast,Not supported,"Reach, trust, fatigue, and belief outputs are not calibrated predictive probabilities."
Intervention efficacy,Not supported externally,An intervention that works inside CHORUS requires empirical evaluation before any real-world claim.


### System context

A C4-style context view places CHORUS inside the actual interaction and publication environment. It distinguishes the player, the application, local persistence, scholarly publication, and assurance outputs without presenting any of them as a hidden remote service.

In [3]:
nodes=[
 dnode('player',40,250,180,110,'Player','Occupies fictional seats; inspects records; chooses actions','person','rect','person'),
 dnode('chorus',360,180,300,250,'CHORUS application','Local deterministic generation, concurrent runtime, disclosure, and receipts','system','rect','software system'),
 dnode('save',850,70,250,110,'Local save / export','Session memory, consented slots, portable text','data','document','local boundary'),
 dnode('scholar',850,255,250,110,'Scholarly publication','Executed notebooks and styled HTML editions','evidence','document','publication'),
 dnode('evidence',850,440,250,110,'Evidence outputs','Tests, retained runs, manifests, checksums','evidence','document','assurance'),
]
edges=[
 dedge('player','chorus',[(220,305),(360,305)],'flow','interacts', (290,292)),
 dedge('chorus','save',[(660,225),(760,225),(760,125),(850,125)],'data','writes only by choice',(755,178)),
 dedge('chorus','scholar',[(660,305),(850,305)],'evidence','publishes',(755,292)),
 dedge('chorus','evidence',[(660,365),(780,365),(780,495),(850,495)],'evidence','produces',(785,420)),
]
_html = diagram_html('C4 system-context diagram','CHORUS in its local and published environment','The player interacts with one local application. Persistence remains player-directed; scholarly records and assurance outputs are published as inspectable artifacts rather than hidden services.',1200,620,nodes,edges,groups=[dgroup('records',810,35,330,550,'Local and published records')],legend=[('flow','Human interaction'),('data','Local data movement'),('evidence','Publication or assurance evidence')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


C4 system-context diagram  CHORUS in its local and published environment  The player interacts with one local application. Persistence remains player-directed; scholarly records and assurance outputs are published as inspectable artifacts rather than hidden services.     CHORUS in its local and published environment  The player interacts with one local application. Persistence remains player-directed; scholarly records and assurance outputs are published as inspectable artifacts rather than hidden services.                Local and published records      interacts      writes only by choice      publishes      produces     PERSON    Player    Occupies fictional  seats; inspects  records; chooses  actions     SOFTWARE SYSTEM    CHORUS application    Local deterministic generation,  concurrent runtime, disclosure, and  receipts     LOCAL BOUNDARY    Local save / export    Session memory, consented  slots, portable text     PUBLICATION    Scholarly publication    Executed notebooks and styled  HTML editions     ASSURANCE    Evidence outputs    Tests, retained runs,  manifests, checksums         Human interaction      Local data movement      Publication or assurance evidence    Validated: 5 nodes · 4 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Player : Occupies fictional seats; inspects records; chooses actions   CHORUS application : Local deterministic generation, concurrent runtime, disclosure, and receipts   Local save / export : Session memory, consented slots, portable text   Scholarly publication : Executed notebooks and styled HTML editions   Evidence outputs : Tests, retained runs, manifests, checksums   Relationships   Player → CHORUS application (interacts)  CHORUS application → Local save / export (writes only by choice)  CHORUS application → Scholarly publication (publishes)  CHORUS application → Evidence outputs (produces)

## Object of representation

A generated night is the primary modeled system. It contains six concurrent fictional seats, one shared logical clock, four decision beats per room, directed relationships, bounded cross-room effects, and an ordered causal record.

In [4]:
levels = [
    ("Actor / seat", "One generated protagonist in a situated role", "Objectives, stakes, relationships, repertoire, evidence access, discernment, enactment, fatigue"),
    ("Artifact", "One bounded information object encountered in a scene", "Source trace, social fit, channel, social proof, record, inference, unknowns"),
    ("Decision beat", "One choice opportunity at a scheduled point", "Visible options, access requirements, local effect, remote effects, time cost"),
    ("Relationship", "A typed dependency or obligation between a seat and another actor or institution", "Trust, authority, care, accountability, competition, audience, market dependence"),
    ("Room", "One incident trajectory", "Four beats, local state, completion snapshot, later afterimage"),
    ("House / network", "Six rooms sharing time and systemic conditions", "Directed routes, ambient effects, compatible direct crossings, aggregate receipts"),
    ("Run / night", "One seed plus one ordered action history", "Reproducible generated pack, state transitions, and final receipt"),
]
_html = table_html("Units and levels of analysis", ("Level", "Unit", "Modeled contents"), levels, row_headers=True)
assert len(levels) == 7
print("Mapped seven nested units from actor to reproducible run.")
_html

Mapped seven nested units from actor to reproducible run.


Level,Unit,Modeled contents
Actor / seat,One generated protagonist in a situated role,"Objectives, stakes, relationships, repertoire, evidence access, discernment, enactment, fatigue"
Artifact,One bounded information object encountered in a scene,"Source trace, social fit, channel, social proof, record, inference, unknowns"
Decision beat,One choice opportunity at a scheduled point,"Visible options, access requirements, local effect, remote effects, time cost"
Relationship,A typed dependency or obligation between a seat and another actor or institution,"Trust, authority, care, accountability, competition, audience, market dependence"
Room,One incident trajectory,"Four beats, local state, completion snapshot, later afterimage"
House / network,Six rooms sharing time and systemic conditions,"Directed routes, ambient effects, compatible direct crossings, aggregate receipts"
Run / night,One seed plus one ordered action history,"Reproducible generated pack, state transitions, and final receipt"


In [5]:
source_owners = [
    ("Scenario grammar", "app/scenario-generator.ts", "Actors, incidents, language, relationships, scenes, choices, routes, and generation coherence"),
    ("Concurrent state", "app/night-engine.ts", "Clock, choice access, event reduction, propagation, fatigue, completion, afterimages, and replay"),
    ("Disclosure", "app/page.tsx", "What a player may inspect before, during, and after a room; relationship and receipt views"),
    ("Persistence", "app/save-model.ts", "Versioned local and portable state; validation and provenance"),
    ("Assurance", "tests/ and evidence/", "Structural invariants, multi-seed runs, browser observations, and bounded release claims"),
]
_html = table_html("Authoritative implementation owners", ("Concern", "Source owner", "Research relevance"), source_owners, row_headers=True)
print("Source ownership is explicit; this notebook documents rather than replaces those authorities.")
_html

Source ownership is explicit; this notebook documents rather than replaces those authorities.


Concern,Source owner,Research relevance
Scenario grammar,app/scenario-generator.ts,"Actors, incidents, language, relationships, scenes, choices, routes, and generation coherence"
Concurrent state,app/night-engine.ts,"Clock, choice access, event reduction, propagation, fatigue, completion, afterimages, and replay"
Disclosure,app/page.tsx,"What a player may inspect before, during, and after a room; relationship and receipt views"
Persistence,app/save-model.ts,Versioned local and portable state; validation and provenance
Assurance,tests/ and evidence/,"Structural invariants, multi-seed runs, browser observations, and bounded release claims"


### Component architecture

The component view assigns generation, runtime mutation, records, persistence, and disclosure to separate authorities. Connector direction describes data, control, or evidence movement; it does not grant the interface permission to become a second source of truth.

In [6]:
nodes=[
 dnode('actor-gen',60,100,210,90,'Actor generation','Roles, objectives, stakes, relationships, repertoires','component','rect','generation'),
 dnode('scenario-gen',60,230,210,90,'Scenario generator','Incidents, artifacts, choices, directed routes','component','rect','generation'),
 dnode('coherence',60,360,210,90,'Coherence gates','Schema, motivation, affect, relations, ethics','evidence','rect','validation'),
 dnode('night-pack',60,500,210,90,'Validated night pack','Six rooms, one clock, deterministic seed','data','document','immutable input'),
 dnode('night-engine',370,170,210,100,'Room / night engine','Choice access, logical time, event reduction','component','rect','runtime'),
 dnode('cross-effects',370,330,210,100,'Cross-scenario effects','Local receipt plus five bounded remote receipts','component','rect','runtime'),
 dnode('night-state',370,500,210,90,'Concurrent night state','Rooms, decisions, pulses, fatigue, afterimages','data','document','state'),
 dnode('receipts',660,200,160,100,'Interpretation / receipt layer','Typed local and remote effects; ordered causal record','evidence','document','record'),
 dnode('save-model',660,470,160,100,'Save model','Schema, digest, preview, consent','data','document','persistence'),
 dnode('house-guide',980,520,220,100,'House Guide / Field Notes','Progressive help and scholarly records','system','rect','surface'),
 dnode('ui-shell',980,360,220,100,'UI shell','One viewport, room switching, privacy controls','system','rect','surface'),
 dnode('trace',980,200,220,100,'Trace / Relations / Receipts','Bounded inspection and concluding interpretation','system','rect','surface'),
]
edges=[
 dedge('actor-gen','scenario-gen',[(165,190),(165,230)],'data','feeds'),
 dedge('scenario-gen','coherence',[(165,320),(165,360)],'data','candidate'),
 dedge('coherence','night-pack',[(165,450),(165,500)],'evidence','accepts'),
 dedge('night-pack','night-engine',[(270,545),(320,545),(320,220),(370,220)],'data','initializes',(320,385)),
 dedge('night-engine','cross-effects',[(475,270),(475,330)],'flow','accepted action'),
 dedge('cross-effects','night-state',[(475,430),(475,500)],'data','mutates'),
 dedge('cross-effects','receipts',[(580,380),(620,380),(620,250),(660,250)],'evidence','emits',(620,315)),
 dedge('night-state','save-model',[(580,545),(620,545),(620,520),(660,520)],'data','serializes',(620,532)),
 dedge('receipts','trace',[(820,250),(980,250)],'evidence','discloses',(900,237)),
 dedge('save-model','ui-shell',[(820,520),(900,520),(900,410),(980,410)],'data','controls',(900,465)),
 dedge('ui-shell','house-guide',[(1090,460),(1090,520)],'flow','opens'),
 dedge('ui-shell','trace',[(1090,360),(1090,300)],'flow','renders'),
]
_html = diagram_html('Component diagram','CHORUS component architecture','Generation creates a validated immutable night pack; the reducer owns state change; records and persistence remain separate; the interface discloses the same authoritative state through bounded surfaces.',1280,700,nodes,edges,groups=[dgroup('g1',25,55,280,580,'Generation'),dgroup('g2',335,55,280,580,'Runtime'),dgroup('g3',630,55,350,580,'Records and persistence'),dgroup('g4',960,55,280,580,'Player-facing surfaces')],legend=[('data','Data or state'),('flow','Runtime control'),('evidence','Receipt / assurance record')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Component diagram  CHORUS component architecture  Generation creates a validated immutable night pack; the reducer owns state change; records and persistence remain separate; the interface discloses the same authoritative state through bounded surfaces.     CHORUS component architecture  Generation creates a validated immutable night pack; the reducer owns state change; records and persistence remain separate; the interface discloses the same authoritative state through bounded surfaces.                Generation     Runtime     Records and persistence     Player-facing surfaces      feeds      candidate      accepts      initializes      accepted action      mutates      emits      serializes      discloses      controls      opens      renders     GENERATION    Actor generation    Roles, objectives, stakes,  relationships, repertoires     GENERATION    Scenario generator    Incidents, artifacts,  choices, directed routes     VALIDATION    Coherence gates    Schema, motivation,  affect, relations, ethics     IMMUTABLE INPUT    Validated night pack    Six rooms, one clock,  deterministic seed     RUNTIME    Room / night engine    Choice access, logical  time, event reduction     RUNTIME    Cross-scenario effects    Local receipt plus five  bounded remote receipts     STATE    Concurrent night state    Rooms, decisions, pulses,  fatigue, afterimages     RECORD    Interpretation /  receipt layer    Typed local and  remote effects;  ordered causal  record     PERSISTENCE    Save model    Schema, digest,  preview, consent     SURFACE    House Guide / Field Notes    Progressive help and  scholarly records     SURFACE    UI shell    One viewport, room  switching, privacy controls     SURFACE    Trace / Relations /  Receipts    Bounded inspection and  concluding interpretation         Data or state      Runtime control      Receipt / assurance record    Validated: 12 nodes · 12 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Actor generation : Roles, objectives, stakes, relationships, repertoires   Scenario generator : Incidents, artifacts, choices, directed routes   Coherence gates : Schema, motivation, affect, relations, ethics   Validated night pack : Six rooms, one clock, deterministic seed   Room / night engine : Choice access, logical time, event reduction   Cross-scenario effects : Local receipt plus five bounded remote receipts   Concurrent night state : Rooms, decisions, pulses, fatigue, afterimages   Interpretation / receipt layer : Typed local and remote effects; ordered causal record   Save model : Schema, digest, preview, consent   House Guide / Field Notes : Progressive help and scholarly records   UI shell : One viewport, room switching, privacy controls   Trace / Relations / Receipts : Bounded inspection and concluding interpretation   Relationships   Actor generation → Scenario generator (feeds)  Scenario generator → Coherence gates (candidate)  Coherence gates → Validated night pack (accepts)  Validated night pack → Room / night engine (initializes)  Room / night engine → Cross-scenario effects (accepted action)  Cross-scenario effects → Concurrent night state (mutates)  Cross-scenario effects → Interpretation / receipt layer (emits)  Concurrent night state → Save model (serializes)  Interpretation / receipt layer → Trace / Relations / Receipts (discloses)  Save model → UI shell (controls)  UI shell → House Guide / Field Notes (opens)  UI shell → Trace / Relations / Receipts (renders)

### Domain model

The entity and event model separates structural social context from the information, decisions, state variables, and causal records produced during play. This prevents an actor label or presentation cue from standing in for a fact about conduct.

In [7]:
nodes=[
 dnode('night',500,35,240,90,'Night / house','Seeded aggregate containing six concurrent rooms','data','rect','aggregate root'),
 dnode('scenario',80,200,200,90,'Scenario','Incident truth, objective, schedule, communication model','component','rect','entity'),
 dnode('room',80,340,200,90,'Room','Four beats, local state, completion snapshot','component','rect','entity'),
 dnode('actor',80,480,200,90,'Actor / seat','Role, motives, affect, capacities, repertoire','person','rect','entity'),
 dnode('relation',80,620,200,90,'Relation','Typed directed tie: trust, care, authority, market','component','rect','entity'),
 dnode('artifact',820,200,200,90,'Artifact','Bounded information object with source and channel','data','document','entity'),
 dnode('signal',820,340,200,90,'Signal','Trace, fit, social proof, presentation surface','component','rect','value object'),
 dnode('decision',820,480,200,90,'Decision','Choice, access requirements, time cost, policy tags','decision','rect','event'),
 dnode('state',820,620,200,90,'State variables','Reach, blame, common ground, fatigue, enactment','data','rect','state'),
 dnode('receipt',820,760,200,70,'Receipts / logs / ledgers','Typed effects and ordered causal provenance','evidence','document','record'),
]
edges=[
 dedge('night','scenario',[(560,125),(560,160),(180,160),(180,200)],'association','contains 6'),
 dedge('night','artifact',[(680,125),(680,160),(920,160),(920,200)],'association','schedules'),
 dedge('scenario','room',[(180,290),(180,340)],'association','instantiates'),
 dedge('room','actor',[(180,430),(180,480)],'association','occupied by'),
 dedge('actor','relation',[(180,570),(180,620)],'association','participates in'),
 dedge('artifact','signal',[(920,290),(920,340)],'association','carries'),
 dedge('signal','decision',[(920,430),(920,480)],'flow','conditions'),
 dedge('decision','state',[(920,570),(920,620)],'data','updates'),
 dedge('state','receipt',[(920,710),(920,760)],'evidence','recorded as'),
 dedge('room','artifact',[(280,385),(600,385),(600,245),(820,245)],'association','presents',(600,315)),
 dedge('actor','decision',[(280,525),(820,525)],'flow','selects',(550,512)),
 dedge('relation','state',[(280,665),(820,665)],'data','constrains',(550,652)),
]
_html = diagram_html('ERD / domain model','Core CHORUS domain model','Structural entities on the left define the fictional social situation; event and state entities on the right describe what arrives, is chosen, changes, and becomes attributable.',1120,860,nodes,edges,groups=[dgroup('struct',35,150,300,610,'Structural model'),dgroup('event',765,150,300,700,'Event and state model')],legend=[('association','Containment or association'),('flow','Actor/event control'),('data','State mutation'),('evidence','Causal record')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


ERD / domain model  Core CHORUS domain model  Structural entities on the left define the fictional social situation; event and state entities on the right describe what arrives, is chosen, changes, and becomes attributable.     Core CHORUS domain model  Structural entities on the left define the fictional social situation; event and state entities on the right describe what arrives, is chosen, changes, and becomes attributable.                   Structural model     Event and state model      contains 6      schedules      instantiates      occupied by      participates in      carries      conditions      updates      recorded as      presents      selects      constrains     AGGREGATE ROOT    Night / house    Seeded aggregate containing  six concurrent rooms     ENTITY    Scenario    Incident truth,  objective, schedule,  communication model     ENTITY    Room    Four beats, local state,  completion snapshot     ENTITY    Actor / seat    Role, motives, affect,  capacities, repertoire     ENTITY    Relation    Typed directed tie:  trust, care, authority,  market     ENTITY    Artifact    Bounded information  object with source and  channel     VALUE OBJECT    Signal    Trace, fit, social  proof, presentation  surface     EVENT    Decision    Choice, access  requirements, time cost,  policy tags     STATE    State variables    Reach, blame, common  ground, fatigue,  enactment     RECORD    Receipts / logs /  ledgers    Typed effects and  ordered causal  provenance         Containment or association      Actor/event control      State mutation      Causal record    Validated: 10 nodes · 12 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Night / house : Seeded aggregate containing six concurrent rooms   Scenario : Incident truth, objective, schedule, communication model   Room : Four beats, local state, completion snapshot   Actor / seat : Role, motives, affect, capacities, repertoire   Relation : Typed directed tie: trust, care, authority, market   Artifact : Bounded information object with source and channel   Signal : Trace, fit, social proof, presentation surface   Decision : Choice, access requirements, time cost, policy tags   State variables : Reach, blame, common ground, fatigue, enactment   Receipts / logs / ledgers : Typed effects and ordered causal provenance   Relationships   Night / house → Scenario (contains 6)  Night / house → Artifact (schedules)  Scenario → Room (instantiates)  Room → Actor / seat (occupied by)  Actor / seat → Relation (participates in)  Artifact → Signal (carries)  Signal → Decision (conditions)  Decision → State variables (updates)  State variables → Receipts / logs / ledgers (recorded as)  Room → Artifact (presents)  Actor / seat → Decision (selects)  Relation → State variables (constrains)

## Construct and variable register

Variables exist to protect a specific distinction or make a specific mechanism inspectable. Most numeric state is a **bounded internal index**, not a naturally measured quantity. Labels such as 0–100 improve consistency and comparison inside the simulation; they do not imply interval validity in the world outside it.

In [8]:
variables = [
    ("seed", "Run control", "Unsigned integer", "Reproduce one generated night without treating it as a representative sample."),
    ("logical time", "Temporal control", "Minutes from house opening", "Order arrivals and effects independently of reading speed or room navigation."),
    ("actor role", "Actor context", "Categorical", "Situate authority, obligations, resources, and audiences without reducing the actor to personality."),
    ("objective", "Actor motive", "Authored text + choice constraints", "Give local action a concrete purpose that may diverge from network-level outcomes."),
    ("stakes", "Actor motive", "Typed material and relational losses", "Explain why a locally plausible shortcut or repair has unequal cost."),
    ("relationships", "Relational structure", "Typed directed ties", "Represent care, authority, trust, dependence, competition, accountability, and audience exposure."),
    ("evidence access", "Epistemic state", "Artifact/source requirements", "Prevent actors from acting on information they do not possess."),
    ("observable record", "Evidence layer", "Bounded propositions", "Separate represented conduct from interpretation."),
    ("room reading", "Interpretive layer", "Authored inference hints", "Represent what the seat makes the record mean without promoting it to fact."),
    ("unknowns", "Uncertainty layer", "Explicit unresolved propositions", "Keep motive, cause, and scope unresolved until supported."),
    ("source trace", "Artifact condition", "Bounded index", "Represent how much source, time, boundary, and context remain attached."),
    ("social fit", "Artifact condition", "Bounded index", "Represent how readily an artifact feels native to a receiving room."),
    ("social proof", "Artifact condition", "Categorical cue", "Represent popularity or endorsement cues without treating them as truth."),
    ("linguistic repertoire", "Communication", "Actor-level set of situated registers", "Allow code choice without essentializing region, class, profession, or identity."),
    ("world model", "Communication", "Assumptions about care, evidence, authority, disagreement, responsibility", "Allow shared language to conceal divergent expectations and different language to express common action."),
    ("presentation temperature", "Communication", "Warm/cool bounded surface", "Separate delivery style from care, honesty, or interior motive."),
    ("discernment", "Capacity", "Bounded internal index", "Represent recognition of a sound action independently of ability to carry it."),
    ("enactment", "Capacity", "Bounded internal index", "Represent remaining follow-through under structural and accumulated load."),
    ("attentional fatigue", "Load", "Bounded internal index", "Represent competing artifacts and context switching."),
    ("affective fatigue", "Load", "Bounded internal index", "Represent repeated urgency, outrage, and anticipatory threat."),
    ("relational fatigue", "Load", "Bounded internal index", "Represent continuous calculation of tone, loyalty, and reply cost."),
    ("verification fatigue", "Load", "Bounded internal index", "Represent source recovery and comparison across fragmented copies."),
    ("efficacy fatigue", "Load", "Bounded internal index", "Represent the sense that careful repair cannot catch a moving cascade."),
    ("reach", "Network outcome", "Synthetic impression count", "Expose scale and overlap inside the authored propagation model."),
    ("blame concentration", "Social outcome", "Bounded internal index", "Represent how distributed failure becomes personalized around a target."),
    ("interpretation gap", "Epistemic outcome", "Bounded internal index", "Represent distance between record and dominant room reading."),
    ("common ground visible", "Coordination outcome", "Bounded internal index", "Represent whether materially shared action remains legible across codes or coalitions."),
    ("active-question focus", "Conversation outcome", "Bounded internal index", "Represent whether the original bounded question remains answerable."),
    ("perceived consensus", "Social outcome", "Bounded internal index", "Represent how repeated or fluent signals alter the apparent social majority."),
    ("effect receipt", "Causal record", "Typed local or remote delta", "Preserve provenance for every accepted decision and scheduled pulse."),
]
_html = table_html("Construct and variable register", ("Variable", "Family", "Representation", "Why it exists"), variables, row_headers=True)
assert len(variables) == 30
assert all(row[3].strip() for row in variables)
print("PASS: 30 declared variables each carry an explicit modeling rationale.")
_html

PASS: 30 declared variables each carry an explicit modeling rationale.


Variable,Family,Representation,Why it exists
seed,Run control,Unsigned integer,Reproduce one generated night without treating it as a representative sample.
logical time,Temporal control,Minutes from house opening,Order arrivals and effects independently of reading speed or room navigation.
actor role,Actor context,Categorical,"Situate authority, obligations, resources, and audiences without reducing the actor to personality."
objective,Actor motive,Authored text + choice constraints,Give local action a concrete purpose that may diverge from network-level outcomes.
stakes,Actor motive,Typed material and relational losses,Explain why a locally plausible shortcut or repair has unequal cost.
relationships,Relational structure,Typed directed ties,"Represent care, authority, trust, dependence, competition, accountability, and audience exposure."
evidence access,Epistemic state,Artifact/source requirements,Prevent actors from acting on information they do not possess.
observable record,Evidence layer,Bounded propositions,Separate represented conduct from interpretation.
room reading,Interpretive layer,Authored inference hints,Represent what the seat makes the record mean without promoting it to fact.
unknowns,Uncertainty layer,Explicit unresolved propositions,"Keep motive, cause, and scope unresolved until supported."


In [9]:
families = {}
for _, family, _, _ in variables:
    families[family] = families.get(family, 0) + 1
cards = [(family, count, "declared constructs") for family, count in sorted(families.items(), key=lambda item: (-item[1], item[0]))]
_html = cards_html("Variable families", cards)
print(f"Coverage check: {len(variables)} variables span {len(families)} construct families.")
_html

Coverage check: 30 variables span 19 construct families.


Load  5  declared constructs    Artifact condition  3  declared constructs    Communication  3  declared constructs    Actor motive  2  declared constructs    Capacity  2  declared constructs    Social outcome  2  declared constructs    Actor context  1  declared constructs    Causal record  1  declared constructs    Conversation outcome  1  declared constructs    Coordination outcome  1  declared constructs    Epistemic outcome  1  declared constructs    Epistemic state  1  declared constructs    Evidence layer  1  declared constructs    Interpretive layer  1  declared constructs    Network outcome  1  declared constructs    Relational structure  1  declared constructs    Run control  1  declared constructs    Temporal control  1  declared constructs    Uncertainty layer  1  declared constructs

## Protected separations

The model is defined as much by what it refuses to collapse as by what it computes.

In [10]:
separations = [
    ("Event record", "Character judgment", "Observed or authored conduct does not automatically establish a stable trait."),
    ("Motive", "Truth", "A defensive or benevolent motive neither falsifies nor validates a claim."),
    ("Explanation", "Exoneration", "Pressure may explain access or choice without transferring responsibility to the harmed party."),
    ("Warmth / reserve", "Care / contempt", "Presentation temperature is not a direct measure of interior orientation."),
    ("Register", "Belief or identity essence", "A learned communication resource does not determine ideology, competence, or moral worth."),
    ("Discernment", "Enactment", "A seat may know a better action while lacking modeled capacity or support to perform it."),
    ("Ambient influence", "Direct content transmission", "Shared social conditions never imply that the same claim crossed rooms."),
    ("Synthetic frequency", "Population prevalence", "Grammar output counts are not survey or observational estimates."),
    ("Coherence", "Truth or predictive validity", "Internal consistency does not establish correspondence with the external world."),
]
_html = table_html("Non-collapsing model commitments", ("Kept separate", "From", "Reason"), separations, row_headers=True)
print("PASS: nine protected separations constrain interpretation of every output.")
_html

PASS: nine protected separations constrain interpretation of every output.


Kept separate,From,Reason
Event record,Character judgment,Observed or authored conduct does not automatically establish a stable trait.
Motive,Truth,A defensive or benevolent motive neither falsifies nor validates a claim.
Explanation,Exoneration,Pressure may explain access or choice without transferring responsibility to the harmed party.
Warmth / reserve,Care / contempt,Presentation temperature is not a direct measure of interior orientation.
Register,Belief or identity essence,"A learned communication resource does not determine ideology, competence, or moral worth."
Discernment,Enactment,A seat may know a better action while lacking modeled capacity or support to perform it.
Ambient influence,Direct content transmission,Shared social conditions never imply that the same claim crossed rooms.
Synthetic frequency,Population prevalence,Grammar output counts are not survey or observational estimates.
Coherence,Truth or predictive validity,Internal consistency does not establish correspondence with the external world.


## Assumptions

Assumptions are explicit design commitments, not hidden facts about human beings. A research variant may alter them, but a report must name the change.

In [11]:
assumptions = [
    ("Bounded fictional interior", "The occupied seat has authored motives and pressures available to the player.", "Needed for perspective-taking; unavailable as an inference about real people."),
    ("Fixed incident truth", "Each generated incident has a stable record that play cannot rewrite.", "Allows evaluation of interpretation drift without rewarding a preferred choice by changing facts."),
    ("Situated rationality", "Choices can be locally plausible given objective, stakes, knowledge, relationships, and load.", "Prevents a simple informed/uninformed or good/bad actor split."),
    ("Typed propagation", "Every remote consequence follows a declared route; direct content requires compatibility.", "Prevents atmospheric influence from being mislabeled as rumor transmission."),
    ("Shared logical clock", "All rooms advance under one event schedule.", "Preserves concurrency and order-independent arrivals."),
    ("Bounded indices", "Internal state is finite and clamped.", "Supports deterministic comparison and guards against runaway state; does not establish empirical scale."),
    ("Non-amplification floor", "Every beat retains an action that does not require repeating or personalizing the claim.", "Avoids coercing the player into modeled harm for progress."),
    ("Sparse diagnostic lenses", "Named critical interpretations appear only when generated evidence and power conditions support them.", "Avoids turning every conflict into the same lesson."),
    ("No player-speed penalty", "Wall-clock reading time and assistive technology do not change modeled fatigue or outcomes.", "Separates accessibility from the represented platform load."),
]
_html = table_html("Model assumptions and purposes", ("Assumption", "Declaration", "Purpose and limit"), assumptions, row_headers=True)
assert len(assumptions) == 9
print("Nine assumptions are explicit and available for ablation or sensitivity testing.")
_html

Nine assumptions are explicit and available for ablation or sensitivity testing.


Assumption,Declaration,Purpose and limit
Bounded fictional interior,The occupied seat has authored motives and pressures available to the player.,Needed for perspective-taking; unavailable as an inference about real people.
Fixed incident truth,Each generated incident has a stable record that play cannot rewrite.,Allows evaluation of interpretation drift without rewarding a preferred choice by changing facts.
Situated rationality,"Choices can be locally plausible given objective, stakes, knowledge, relationships, and load.",Prevents a simple informed/uninformed or good/bad actor split.
Typed propagation,Every remote consequence follows a declared route; direct content requires compatibility.,Prevents atmospheric influence from being mislabeled as rumor transmission.
Shared logical clock,All rooms advance under one event schedule.,Preserves concurrency and order-independent arrivals.
Bounded indices,Internal state is finite and clamped.,Supports deterministic comparison and guards against runaway state; does not establish empirical scale.
Non-amplification floor,Every beat retains an action that does not require repeating or personalizing the claim.,Avoids coercing the player into modeled harm for progress.
Sparse diagnostic lenses,Named critical interpretations appear only when generated evidence and power conditions support them.,Avoids turning every conflict into the same lesson.
No player-speed penalty,Wall-clock reading time and assistive technology do not change modeled fatigue or outcomes.,Separates accessibility from the represented platform load.


In [12]:
exclusions = [
    ("No calibrated behavioral probabilities", "Choice availability and outcomes are authored mechanisms, not estimated propensities."),
    ("No demographic essentialism", "Age, class, region, language, coalition, or occupation never determines motive or conduct by itself."),
    ("No clinical inference", "Fatigue and pressure channels are interaction constructs, not diagnoses."),
    ("No truth-by-consensus", "Reach, fluency, repetition, or perceived agreement never changes ground truth."),
    ("No universal network topology", "Six rooms and their routes are a designed laboratory, not a model of every platform."),
    ("No complete institutional model", "Authorization, market, family, peer, and political relations are bounded role structures rather than full organizations."),
    ("No claim of ecological completeness", "Offline events, private channels, media systems, history, and material conditions are represented selectively."),
]
_html = checklist_html("Deliberate exclusions", [("OUTSIDE SCOPE", label, note) for label, note in exclusions])
print("Scope check: seven high-risk overclaims are explicitly excluded.")
_html

Scope check: seven high-risk overclaims are explicitly excluded.


OUTSIDE SCOPE  No calibrated behavioral probabilities  Choice availability and outcomes are authored mechanisms, not estimated propensities.    OUTSIDE SCOPE  No demographic essentialism  Age, class, region, language, coalition, or occupation never determines motive or conduct by itself.    OUTSIDE SCOPE  No clinical inference  Fatigue and pressure channels are interaction constructs, not diagnoses.    OUTSIDE SCOPE  No truth-by-consensus  Reach, fluency, repetition, or perceived agreement never changes ground truth.    OUTSIDE SCOPE  No universal network topology  Six rooms and their routes are a designed laboratory, not a model of every platform.    OUTSIDE SCOPE  No complete institutional model  Authorization, market, family, peer, and political relations are bounded role structures rather than full organizations.    OUTSIDE SCOPE  No claim of ecological completeness  Offline events, private channels, media systems, history, and material conditions are represented selectively.

## Mechanism chain

A CHORUS result is not generated by a single misinformation score. It emerges from typed transitions between evidence, interpretation, relationships, capacity, and propagation.

In [13]:
mechanism = [
    (1, "Generated context", "Role, objective, stakes, repertoire, relationships, and incident truth are established."),
    (2, "Artifact arrival", "The shared clock exposes a bounded artifact with trace, fit, channel, and social proof."),
    (3, "Situated reading", "The seat separates record, local interpretation, and unresolved questions."),
    (4, "Choice access", "Evidence, authority, relationships, assembled support, enactment, and the non-amplification floor determine available action."),
    (5, "Accepted decision", "The reducer advances logical time and writes one source-room event."),
    (6, "Typed receipts", "One local and five remote consequences are recorded; direct content is permitted only on compatible routes."),
    (7, "State update", "Reach, social conditions, fatigue, capacity, and later access change within bounds."),
    (8, "Afterimage / conclusion", "Closed rooms retain completion snapshots while later effects remain visible; the whole-night receipt resolves authored interpretation."),
]
_html = table_html("Mechanism from context to afterimage", ("Step", "Stage", "Transition"), mechanism)
assert [row[0] for row in mechanism] == list(range(1, 9))
print("PASS: eight-stage mechanism preserves a complete causal path from generation to receipt.")
_html

PASS: eight-stage mechanism preserves a complete causal path from generation to receipt.


Step,Stage,Transition
1,Generated context,"Role, objective, stakes, repertoire, relationships, and incident truth are established."
2,Artifact arrival,"The shared clock exposes a bounded artifact with trace, fit, channel, and social proof."
3,Situated reading,"The seat separates record, local interpretation, and unresolved questions."
4,Choice access,"Evidence, authority, relationships, assembled support, enactment, and the non-amplification floor determine available action."
5,Accepted decision,The reducer advances logical time and writes one source-room event.
6,Typed receipts,One local and five remote consequences are recorded; direct content is permitted only on compatible routes.
7,State update,"Reach, social conditions, fatigue, capacity, and later access change within bounds."
8,Afterimage / conclusion,Closed rooms retain completion snapshots while later effects remain visible; the whole-night receipt resolves authored interpretation.


### Concurrent-night activity flow

The activity diagram shows where navigation ends and mutation begins. It makes blocked choices, parallel local and remote effects, validation, receipt generation, looping work, and the final interpretation gate explicit.

In [14]:
nodes=[
 dnode('start',420,25,200,60,'Start generated night','Seed and reviewed pack available','system','pill','start'),
 dnode('init',380,120,280,80,'Night initialization','Create shared clock and six room runtimes','component','rect','activity'),
 dnode('enter',380,240,280,80,'Enter or resume a room','Navigation does not advance time','component','rect','activity'),
 dnode('attempt',380,360,280,80,'Attempt a decision','Current beat and visible choice selected','decision','rect','activity'),
 dnode('access',430,480,180,100,'Access passes?','Evidence, structure, enactment','decision','diamond','decision'),
 dnode('blocked',80,500,220,90,'Explain unavailable path','Name motive, pressure, and capacity; do not mutate','boundary','rect','alternate'),
 dnode('accepted',380,630,280,80,'Accept event','Advance modeled time exactly once','component','rect','activity'),
 dnode('local',180,760,240,80,'Apply local update','Source room state and fatigue','data','rect','parallel activity'),
 dnode('remote',620,760,240,80,'Propagate remote effects','Five bounded directed receipts','data','rect','parallel activity'),
 dnode('coherence',380,890,280,80,'Validate transition','Bounds, causality, compatibility, replay','evidence','rect','gate'),
 dnode('receipts',380,1010,280,80,'Generate causal receipts','Local, remote, scheduled, and afterimage records','evidence','document','activity'),
 dnode('more',430,1130,180,100,'More beats?','Any room has due work','decision','diamond','decision'),
 dnode('end',380,1260,280,80,'End-of-night interpretation','Unseal whole-house receipt and research record','system','pill','end'),
]
edges=[
 dedge('start','init',[(520,85),(520,120)],'flow'),
 dedge('init','enter',[(520,200),(520,240)],'flow'),
 dedge('enter','attempt',[(520,320),(520,360)],'flow'),
 dedge('attempt','access',[(520,440),(520,480)],'flow'),
 dedge('access','blocked',[(430,530),(340,530),(340,545),(300,545)],'boundary','no',(345,515)),
 dedge('blocked','attempt',[(80,545),(40,545),(40,400),(380,400)],'boundary','retry',(210,387)),
 dedge('access','accepted',[(520,580),(520,630)],'flow','yes',(555,605)),
 dedge('accepted','local',[(450,710),(450,730),(300,730),(300,760)],'data','local'),
 dedge('accepted','remote',[(590,710),(590,730),(740,730),(740,760)],'data','remote'),
 dedge('local','coherence',[(300,840),(300,870),(480,870),(480,890)],'evidence'),
 dedge('remote','coherence',[(740,840),(740,870),(560,870),(560,890)],'evidence'),
 dedge('coherence','receipts',[(520,970),(520,1010)],'evidence'),
 dedge('receipts','more',[(520,1090),(520,1130)],'flow'),
 dedge('more','enter',[(610,1180),(950,1180),(950,280),(660,280)],'flow','yes',(945,730)),
 dedge('more','end',[(520,1230),(520,1260)],'flow','no',(555,1245)),
]
_html = diagram_html('UML activity / state-flow diagram','Concurrent-night decision lifecycle','The activity diagram separates navigation, choice access, accepted mutation, parallel local and remote effects, validation, receipt generation, looping work, and the final interpretation gate.',1040,1370,nodes,edges,legend=[('flow','Control flow'),('data','State effect'),('evidence','Validation / receipt'),('boundary','Unavailable path without mutation')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


UML activity / state-flow diagram  Concurrent-night decision lifecycle  The activity diagram separates navigation, choice access, accepted mutation, parallel local and remote effects, validation, receipt generation, looping work, and the final interpretation gate.     Concurrent-night decision lifecycle  The activity diagram separates navigation, choice access, accepted mutation, parallel local and remote effects, validation, receipt generation, looping work, and the final interpretation gate.                        no      retry      yes      local      remote          yes      no     START    Start generated night    Seed and reviewed pack  available     ACTIVITY    Night initialization    Create shared clock and six room  runtimes     ACTIVITY    Enter or resume a room    Navigation does not advance time     ACTIVITY    Attempt a decision    Current beat and visible choice  selected     DECISION    Access passes?    Evidence, structure,  enactment     ALTERNATE    Explain unavailable path    Name motive, pressure, and  capacity; do not mutate     ACTIVITY    Accept event    Advance modeled time exactly once     PARALLEL ACTIVITY    Apply local update    Source room state and fatigue     PARALLEL ACTIVITY    Propagate remote effects    Five bounded directed receipts     GATE    Validate transition    Bounds, causality, compatibility,  replay     ACTIVITY    Generate causal receipts    Local, remote, scheduled, and  afterimage records     DECISION    More beats?    Any room has due work     END    End-of-night interpretation    Unseal whole-house receipt and  research record         Control flow      State effect      Validation / receipt      Unavailable path without mutation    Validated: 13 nodes · 15 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Start generated night : Seed and reviewed pack available   Night initialization : Create shared clock and six room runtimes   Enter or resume a room : Navigation does not advance time   Attempt a decision : Current beat and visible choice selected   Access passes? : Evidence, structure, enactment   Explain unavailable path : Name motive, pressure, and capacity; do not mutate   Accept event : Advance modeled time exactly once   Apply local update : Source room state and fatigue   Propagate remote effects : Five bounded directed receipts   Validate transition : Bounds, causality, compatibility, replay   Generate causal receipts : Local, remote, scheduled, and afterimage records   More beats? : Any room has due work   End-of-night interpretation : Unseal whole-house receipt and research record   Relationships   Start generated night → Night initialization  Night initialization → Enter or resume a room  Enter or resume a room → Attempt a decision  Attempt a decision → Access passes?  Access passes? → Explain unavailable path (no)  Explain unavailable path → Attempt a decision (retry)  Access passes? → Accept event (yes)  Accept event → Apply local update (local)  Accept event → Propagate remote effects (remote)  Apply local update → Validate transition  Propagate remote effects → Validate transition  Validate transition → Generate causal receipts  Generate causal receipts → More beats?  More beats? → Enter or resume a room (yes)  More beats? → End-of-night interpretation (no)

### Propagation pathways

The directed acyclic graph separates trust and social capital, salience and amplification, and accountability pressure with rumor or scapegoat dynamics. These are authored causal pathways inside CHORUS, not calibrated causal estimates for real populations.

In [15]:
nodes=[
 dnode('action',40,310,180,100,'Accepted action','One situated choice at one decision beat','decision','rect','cause'),
 dnode('trust',300,100,220,100,'Trust / social capital','Source standing, dependence, reciprocal access','component','rect','pathway'),
 dnode('salience',300,310,220,100,'Salience','Attention, urgency, repetition, visibility','component','rect','pathway'),
 dnode('accountability',300,520,220,100,'Accountability pressure','Threat, reply asymmetry, status protection','component','rect','pathway'),
 dnode('correction',600,100,220,100,'Correction uptake','Whether source restoration is carried or resisted','component','rect','mechanism'),
 dnode('amplification',600,310,220,100,'Amplification','Reach, perceived consensus, repeated exposure','component','rect','mechanism'),
 dnode('rumor',600,520,220,100,'Rumor / scapegoat dynamics','Personalization, blame concentration, displaced question','component','rect','mechanism'),
 dnode('remote-trust',900,100,220,100,'Cross-room trust conditions','Credibility and repair access elsewhere','data','rect','remote effect'),
 dnode('remote-reach',900,310,220,100,'Cross-room reach conditions','Ambient impressions and consensus pressure','data','rect','remote effect'),
 dnode('remote-blame',900,520,220,100,'Cross-room attribution conditions','Blame, interpretation gap, thread focus','data','rect','remote effect'),
 dnode('house',1200,300,180,120,'House state / later choices','Updated access, fatigue, common ground, and afterimages','system','rect','downstream state'),
]
edges=[
 dedge('action','trust',[(220,335),(260,335),(260,150),(300,150)],'flow'),
 dedge('action','salience',[(220,360),(300,360)],'flow'),
 dedge('action','accountability',[(220,385),(260,385),(260,570),(300,570)],'flow'),
 dedge('trust','correction',[(520,150),(600,150)],'data'),
 dedge('salience','amplification',[(520,360),(600,360)],'data'),
 dedge('accountability','rumor',[(520,570),(600,570)],'data'),
 dedge('correction','remote-trust',[(820,150),(900,150)],'data'),
 dedge('amplification','remote-reach',[(820,360),(900,360)],'data'),
 dedge('rumor','remote-blame',[(820,570),(900,570)],'data'),
 dedge('remote-trust','house',[(1120,150),(1150,150),(1150,345),(1200,345)],'data'),
 dedge('remote-reach','house',[(1120,360),(1200,360)],'data'),
 dedge('remote-blame','house',[(1120,570),(1165,570),(1165,390),(1200,390)],'data'),
]
_html = diagram_html('Directed acyclic causal diagram','Authored propagation pathways','One accepted action can alter three distinct modeled pathways. They remain separate until their bounded remote consequences update the shared house state; arrows describe CHORUS rules, not calibrated real-world causation.',1400,720,nodes,edges,legend=[('flow','Accepted action enters pathway'),('data','Authored state dependency')],notes=['Parallel rows are intentionally non-interchangeable: trust, salience, and accountability pressure are distinct constructs.'])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Directed acyclic causal diagram  Authored propagation pathways  One accepted action can alter three distinct modeled pathways. They remain separate until their bounded remote consequences update the shared house state; arrows describe CHORUS rules, not calibrated real-world causation.     Authored propagation pathways  One accepted action can alter three distinct modeled pathways. They remain separate until their bounded remote consequences update the shared house state; arrows describe CHORUS rules, not calibrated real-world causation.                         CAUSE    Accepted action    One situated choice  at one decision beat     PATHWAY    Trust / social capital    Source standing,  dependence, reciprocal  access     PATHWAY    Salience    Attention, urgency,  repetition, visibility     PATHWAY    Accountability pressure    Threat, reply asymmetry,  status protection     MECHANISM    Correction uptake    Whether source restoration  is carried or resisted     MECHANISM    Amplification    Reach, perceived consensus,  repeated exposure     MECHANISM    Rumor / scapegoat dynamics    Personalization, blame  concentration, displaced  question     REMOTE EFFECT    Cross-room trust conditions    Credibility and repair  access elsewhere     REMOTE EFFECT    Cross-room reach conditions    Ambient impressions and  consensus pressure     REMOTE EFFECT    Cross-room attribution  conditions    Blame, interpretation gap,  thread focus     DOWNSTREAM STATE    House state / later  choices    Updated access,  fatigue, common  ground, and  afterimages         Accepted action enters pathway      Authored state dependency     Parallel rows are intentionally non-interchangeable: trust, salience, and accountability pressure are distinct constructs.   Validated: 11 nodes · 12 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Accepted action : One situated choice at one decision beat   Trust / social capital : Source standing, dependence, reciprocal access   Salience : Attention, urgency, repetition, visibility   Accountability pressure : Threat, reply asymmetry, status protection   Correction uptake : Whether source restoration is carried or resisted   Amplification : Reach, perceived consensus, repeated exposure   Rumor / scapegoat dynamics : Personalization, blame concentration, displaced question   Cross-room trust conditions : Credibility and repair access elsewhere   Cross-room reach conditions : Ambient impressions and consensus pressure   Cross-room attribution conditions : Blame, interpretation gap, thread focus   House state / later choices : Updated access, fatigue, common ground, and afterimages   Relationships   Accepted action → Trust / social capital  Accepted action → Salience  Accepted action → Accountability pressure  Trust / social capital → Correction uptake  Salience → Amplification  Accountability pressure → Rumor / scapegoat dynamics  Correction uptake → Cross-room trust conditions  Amplification → Cross-room reach conditions  Rumor / scapegoat dynamics → Cross-room attribution conditions  Cross-room trust conditions → House state / later choices  Cross-room reach conditions → House state / later choices  Cross-room attribution conditions → House state / later choices

## Coherence evaluation

Coherence is a family of internal consistency tests. It is evaluated across levels rather than collapsed into one persuasive score. A scalar shown in the interface is an inspectable summary of authored gates, not evidence of realism.

In [16]:
coherence = [
    ("Schema", "All required actors, rooms, scenes, choices, routes, ledgers, and receipts exist with valid types and bounds.", "Missing field, unknown enum, non-finite value, incomplete route set"),
    ("Agentic", "An actor's objective, stakes, knowledge, relationships, repertoire, baseline capacities, and choices form a plausible local action space.", "Choice requires absent knowledge; role has no meaningful stake; motivation is only a generic label"),
    ("Motivational", "Immediate choices can be traced to represented motives, incentives, authority, or protection goals.", "Outcome appears because the lesson needs it rather than because the seat has a reason"),
    ("Affective", "Pressure and fatigue accumulate or recede in response to represented events without replacing discernment.", "Emotional state jumps without cause; fatigue becomes diagnosis or moral excuse"),
    ("Relational", "Trust, care, authority, dependence, accountability, competition, and audience ties constrain action and consequence consistently.", "A tie reverses role or obligation without an event; costs fall on an unrelated actor without a route"),
    ("Epistemic", "Record, inference, authored interior, and unknowns remain distinct; actors cannot know sealed information.", "Inference becomes fact; remote room receives unsupported content; final labels leak into active play"),
    ("Temporal", "Arrivals and pulses follow the shared logical clock and occur exactly once regardless of navigation order.", "Visiting creates an event; delayed entry rewrites its arrival; duplicate pulse"),
    ("Causal", "Every accepted decision and autonomous pulse has complete, typed, attributable receipts.", "Missing local effect, fewer or more than five remote effects, stale choice mutates state"),
    ("Network", "Ambient effects and direct crossings respect route direction and compatibility.", "Shared atmosphere is described as shared content; direct crossing lacks carrier compatibility"),
    ("Narrative", "Six incidents are distinct yet mutually consequential, with continuity across all four beats and the closing receipt.", "Actor changes identity to satisfy a scene; scenario becomes a fixed morality vignette"),
    ("Ethical", "Youth safety, non-amplification, diagnostic restraint, responsibility, and correction boundaries remain intact.", "Player must spread harm; style or identity is used as proof of motive"),
    ("Reproducibility", "Seed plus ordered action history reconstructs the same generated pack and final state.", "Replay depends on visit order, wall-clock delay, or unrecorded randomness"),
]
_html = table_html("Multidimensional coherence rubric", ("Dimension", "Pass condition", "Failure signal"), coherence, row_headers=True)
assert len(coherence) == 12
print("PASS: coherence is evaluated across 12 independent dimensions rather than one realism score.")
_html

PASS: coherence is evaluated across 12 independent dimensions rather than one realism score.


Dimension,Pass condition,Failure signal
Schema,"All required actors, rooms, scenes, choices, routes, ledgers, and receipts exist with valid types and bounds.","Missing field, unknown enum, non-finite value, incomplete route set"
Agentic,"An actor's objective, stakes, knowledge, relationships, repertoire, baseline capacities, and choices form a plausible local action space.",Choice requires absent knowledge; role has no meaningful stake; motivation is only a generic label
Motivational,"Immediate choices can be traced to represented motives, incentives, authority, or protection goals.",Outcome appears because the lesson needs it rather than because the seat has a reason
Affective,Pressure and fatigue accumulate or recede in response to represented events without replacing discernment.,Emotional state jumps without cause; fatigue becomes diagnosis or moral excuse
Relational,"Trust, care, authority, dependence, accountability, competition, and audience ties constrain action and consequence consistently.",A tie reverses role or obligation without an event; costs fall on an unrelated actor without a route
Epistemic,"Record, inference, authored interior, and unknowns remain distinct; actors cannot know sealed information.",Inference becomes fact; remote room receives unsupported content; final labels leak into active play
Temporal,Arrivals and pulses follow the shared logical clock and occur exactly once regardless of navigation order.,Visiting creates an event; delayed entry rewrites its arrival; duplicate pulse
Causal,"Every accepted decision and autonomous pulse has complete, typed, attributable receipts.","Missing local effect, fewer or more than five remote effects, stale choice mutates state"
Network,Ambient effects and direct crossings respect route direction and compatibility.,Shared atmosphere is described as shared content; direct crossing lacks carrier compatibility
Narrative,"Six incidents are distinct yet mutually consequential, with continuity across all four beats and the closing receipt.",Actor changes identity to satisfy a scene; scenario becomes a fixed morality vignette


In [17]:
coherence_workflow = [
    ("Generation gate", "Reject malformed or contradictory packs before play", "Schema, agentic, narrative, ethical"),
    ("Reducer invariant", "Reject stale, duplicate, unsupported, or non-finite transitions", "Temporal, causal, network, reproducibility"),
    ("Multi-seed harness", "Exercise many deterministic nights and choice policies", "Coverage, bounds, exhaustion, replay"),
    ("Disclosure test", "Inspect which information is visible at each stage", "Epistemic and ethical"),
    ("Close reading", "Review motives, relationships, affect, language, and continuity", "Agentic, motivational, affective, relational, narrative"),
    ("Variant audit", "Change one assumption and compare matched seeds", "Mechanism sensitivity and hidden coupling"),
]
_html = table_html("Coherence evaluation workflow", ("Layer", "Procedure", "Primary dimensions"), coherence_workflow, row_headers=True)
print("Coherence evaluation combines executable gates with qualitative inspection.")
_html

Coherence evaluation combines executable gates with qualitative inspection.


Layer,Procedure,Primary dimensions
Generation gate,Reject malformed or contradictory packs before play,"Schema, agentic, narrative, ethical"
Reducer invariant,"Reject stale, duplicate, unsupported, or non-finite transitions","Temporal, causal, network, reproducibility"
Multi-seed harness,Exercise many deterministic nights and choice policies,"Coverage, bounds, exhaustion, replay"
Disclosure test,Inspect which information is visible at each stage,Epistemic and ethical
Close reading,"Review motives, relationships, affect, language, and continuity","Agentic, motivational, affective, relational, narrative"
Variant audit,Change one assumption and compare matched seeds,Mechanism sensitivity and hidden coupling


## Validity framework

Internal consistency is necessary but insufficient. Each validity domain asks a different question and requires different evidence.

In [18]:
validity = [
    ("Construct validity", "Do variables represent the distinctions the thesis claims?", "Definition audit, discriminant checks, qualitative review, expert critique", "A bounded index may remain a useful design construct without being a validated psychometric scale."),
    ("Internal validity", "Does a changed model condition cause the changed model outcome?", "Matched-seed variants, controlled policies, ablation, event-receipt tracing", "Supports causality only inside the authored model."),
    ("External validity", "Do findings generalize beyond CHORUS?", "Independent empirical data and replication across settings", "Currently not established."),
    ("Ecological validity", "Does the experience resemble relevant information environments?", "User studies, domain review, task comparison, field observation", "Atmospheric plausibility alone is insufficient."),
    ("Content validity", "Does the grammar cover the relevant mechanism space?", "Expert mapping, missing-case analysis, scenario diversity review", "Six rooms are intentionally selective."),
    ("Statistical conclusion validity", "Are reported differences stable and estimated appropriately?", "Predeclared seed domains, effect distributions, uncertainty, multiple-comparison control", "Synthetic sample size does not substitute for population data."),
    ("Ethical validity", "Are interpretations and uses consistent with the model's responsibility boundaries?", "Misuse analysis, disclosure review, youth safeguards, human-subject protocol when applicable", "A technically reproducible result can still be an invalid use."),
]
_html = table_html("Validity domains and evidence requirements", ("Domain", "Question", "Needed evidence", "Current boundary"), validity, row_headers=True)
print("Validity matrix distinguishes seven questions that cannot be answered by one coherence score.")
_html

Validity matrix distinguishes seven questions that cannot be answered by one coherence score.


Domain,Question,Needed evidence,Current boundary
Construct validity,Do variables represent the distinctions the thesis claims?,"Definition audit, discriminant checks, qualitative review, expert critique",A bounded index may remain a useful design construct without being a validated psychometric scale.
Internal validity,Does a changed model condition cause the changed model outcome?,"Matched-seed variants, controlled policies, ablation, event-receipt tracing",Supports causality only inside the authored model.
External validity,Do findings generalize beyond CHORUS?,Independent empirical data and replication across settings,Currently not established.
Ecological validity,Does the experience resemble relevant information environments?,"User studies, domain review, task comparison, field observation",Atmospheric plausibility alone is insufficient.
Content validity,Does the grammar cover the relevant mechanism space?,"Expert mapping, missing-case analysis, scenario diversity review",Six rooms are intentionally selective.
Statistical conclusion validity,Are reported differences stable and estimated appropriately?,"Predeclared seed domains, effect distributions, uncertainty, multiple-comparison control",Synthetic sample size does not substitute for population data.
Ethical validity,Are interpretations and uses consistent with the model's responsibility boundaries?,"Misuse analysis, disclosure review, youth safeguards, human-subject protocol when applicable",A technically reproducible result can still be an invalid use.


In [19]:
interpretation_rules = [
    ("Describe", "Report the generated condition and model output without human attribution."),
    ("Compare", "Compare matched variants or policies under the same seed domain."),
    ("Explain internally", "Trace the difference through typed events and receipts."),
    ("Qualify", "Name assumptions, uncertainty, sensitivity, and finite coverage."),
    ("Do not generalize", "Stop before claims about real people, prevalence, prediction, or intervention efficacy."),
    ("Validate externally", "Use empirical research before crossing the model/world boundary."),
]
_html = table_html("Permitted interpretation sequence", ("Stage", "Required practice"), interpretation_rules, row_headers=True)
print("Interpretation sequence contains an explicit stop before external generalization.")
_html

Interpretation sequence contains an explicit stop before external generalization.


Stage,Required practice
Describe,Report the generated condition and model output without human attribution.
Compare,Compare matched variants or policies under the same seed domain.
Explain internally,Trace the difference through typed events and receipts.
Qualify,"Name assumptions, uncertainty, sensitivity, and finite coverage."
Do not generalize,"Stop before claims about real people, prevalence, prediction, or intervention efficacy."
Validate externally,Use empirical research before crossing the model/world boundary.


## Sensitivity and uncertainty priorities

The model is most informative when its conclusions survive reasonable alternatives—or when it clearly reveals which assumptions carry them.

In [20]:
sensitivity = [
    ("Propagation magnitude", "Sweep local and remote effect sizes", "Does the qualitative ordering of outcomes persist?"),
    ("Network topology", "Remove, add, or reweight directed routes", "Is a result an artifact of one six-room graph?"),
    ("Correction timing", "Advance or delay source restoration", "Which outcomes depend on order rather than content?"),
    ("Artifact fit and trace", "Vary independently", "Does fluency overwhelm provenance only under selected thresholds?"),
    ("Fatigue composition", "Hold total load constant while reallocating channels", "Which capacity barriers depend on load type?"),
    ("Baseline enactment", "Sweep actor carrying power", "Does one actor template dominate the result?"),
    ("Choice policy", "Compare non-amplifying, verification-first, relationship-first, and randomized policies", "Are findings robust to the decision rule?"),
    ("Generation grammar", "Ablate one communication dynamic or role family", "Does a claimed mechanism require a particular narrative assignment?"),
]
_html = table_html("Priority sensitivity analyses", ("Target", "Variant", "Question"), sensitivity, row_headers=True)
print("Eight sensitivity targets identify the assumptions most likely to drive a result.")
_html

Eight sensitivity targets identify the assumptions most likely to drive a result.


Target,Variant,Question
Propagation magnitude,Sweep local and remote effect sizes,Does the qualitative ordering of outcomes persist?
Network topology,"Remove, add, or reweight directed routes",Is a result an artifact of one six-room graph?
Correction timing,Advance or delay source restoration,Which outcomes depend on order rather than content?
Artifact fit and trace,Vary independently,Does fluency overwhelm provenance only under selected thresholds?
Fatigue composition,Hold total load constant while reallocating channels,Which capacity barriers depend on load type?
Baseline enactment,Sweep actor carrying power,Does one actor template dominate the result?
Choice policy,"Compare non-amplifying, verification-first, relationship-first, and randomized policies",Are findings robust to the decision rule?
Generation grammar,Ablate one communication dynamic or role family,Does a claimed mechanism require a particular narrative assignment?


## Ethical and maintenance contract

The model specification is part of the executable release surface. A new variable, actor grammar, relationship type, propagation rule, or interpretive label requires a rationale, authority, bounds, disclosure timing, tests, and a claim-limit review.

In [21]:
change_contract = [
    ("Definition", "Name the construct and distinguish it from adjacent constructs."),
    ("Rationale", "State why the variable exists and which project thesis or invariant it protects."),
    ("Operationalization", "Declare type, bounds, units or index status, initialization, and update rules."),
    ("Causal authority", "Name the module allowed to create or mutate it."),
    ("Disclosure", "Specify what the player may see before, during, and after the whole night."),
    ("Validation", "Add structural, transition, replay, and adverse-input checks where applicable."),
    ("Sensitivity", "Identify plausible alternatives and whether conclusions depend on them."),
    ("Ethics", "Review diagnosis, identity essentialism, youth safety, coercion, privacy, and misuse."),
    ("Documentation", "Update the canonical prose owner, notebook, traceability map, and decision record."),
]
_html = checklist_html("Model-change completion contract", [("REQUIRED", label, evidence) for label, evidence in change_contract])
print("Model maintenance requires nine companions; code alone is not a complete change.")
_html

Model maintenance requires nine companions; code alone is not a complete change.


REQUIRED  Definition  Name the construct and distinguish it from adjacent constructs.    REQUIRED  Rationale  State why the variable exists and which project thesis or invariant it protects.    REQUIRED  Operationalization  Declare type, bounds, units or index status, initialization, and update rules.    REQUIRED  Causal authority  Name the module allowed to create or mutate it.    REQUIRED  Disclosure  Specify what the player may see before, during, and after the whole night.    REQUIRED  Validation  Add structural, transition, replay, and adverse-input checks where applicable.    REQUIRED  Sensitivity  Identify plausible alternatives and whether conclusions depend on them.    REQUIRED  Ethics  Review diagnosis, identity essentialism, youth safety, coercion, privacy, and misuse.    REQUIRED  Documentation  Update the canonical prose owner, notebook, traceability map, and decision record.

## Specification conclusion

CHORUS is strongest when read as a transparent mechanism laboratory: rich enough to preserve individual motive, affect, relationship, language, institutional constraint, and network consequence; bounded enough that its fictional outputs are never mistaken for measurements of real people.

The companion **Research Design Atlas** turns these constructs into testable within-model questions and defines what additional evidence would be required for any empirical extension.